# Ingest chunks into PostgreSQL pgvector

Loads pre-computed embeddings from `data/chunks/chunks_collection.jsonl` and ingests them into PostgreSQL using `langchain-postgres` PGVector.

**Prerequisites:** Docker container `pgvector-rag` must be running (`docker start pgvector-rag`).

In [1]:
import json
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_postgres import PGVector

load_dotenv()

POSTGRES_URL = os.environ["POSTGRES_URL"]
OPENAI_KEY = os.environ["OPENAI_KEY"]
CHUNKS_PATH = Path("data/chunks/chunks_collection.jsonl")
COLLECTION_NAME = "sec_10k_chunks"

## Load chunks from JSONL

In [3]:
chunks = []
with open(CHUNKS_PATH, encoding="utf-8") as f:
    for line in f:
        c = json.loads(line)
        # rename key: "stock symbol" (with space) is not a valid filter identifier
        if "stock symbol" in c["metadata"]:
            c["metadata"]["stock_symbol"] = c["metadata"].pop("stock symbol")
        chunks.append(c)

print(f"Loaded {len(chunks)} chunks")
print(f"Example chunk_id: {chunks[0]['chunk_id']}")
print(f"Metadata keys: {list(chunks[0]['metadata'].keys())}")

Loaded 6702 chunks
Example chunk_id: aapl-20250927_section_1_chunk_1
Metadata keys: ['chunk_idx', 'source', 'doc_id', 'emb_model', 'fiscal_year_end', 'filing_date', 'part', 'section', 'chunk', 'stock_symbol']


## Initialize PGVector store

This creates the `langchain_pg_collection` and `langchain_pg_embedding` tables if they don't exist.
We pass an `OpenAIEmbeddings` object because PGVector requires one, but we won't use it to re-embed — we'll call `add_embeddings` directly with our pre-computed vectors.

In [4]:
embeddings_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=OPENAI_KEY,
)

store = PGVector(
    connection=POSTGRES_URL,
    embeddings=embeddings_model,
    collection_name=COLLECTION_NAME,
    use_jsonb=True,
)

print("PGVector store initialized")

PGVector store initialized


## Ingest chunks

Drops any existing collection first (so re-runs are safe), then inserts all chunks.
Uses `add_embeddings` to insert pre-computed vectors — no OpenAI API calls made.

In [5]:
store.delete_collection()
store.create_collection()
print("Collection reset")

BATCH_SIZE = 500

ids_inserted = []
for i in range(0, len(chunks), BATCH_SIZE):
    batch = chunks[i : i + BATCH_SIZE]
    ids = store.add_embeddings(
        texts=[c["text"] for c in batch],
        embeddings=[c["embedding"] for c in batch],
        metadatas=[c["metadata"] for c in batch],
        ids=[c["chunk_id"] for c in batch],
    )
    ids_inserted.extend(ids)
    print(f"Inserted batch {i // BATCH_SIZE + 1}: chunks {i}–{i + len(batch) - 1}")

print(f"\nDone. Total inserted: {len(ids_inserted)}")

Collection reset
Inserted batch 1: chunks 0–499
Inserted batch 2: chunks 500–999
Inserted batch 3: chunks 1000–1499
Inserted batch 4: chunks 1500–1999
Inserted batch 5: chunks 2000–2499
Inserted batch 6: chunks 2500–2999
Inserted batch 7: chunks 3000–3499
Inserted batch 8: chunks 3500–3999
Inserted batch 9: chunks 4000–4499
Inserted batch 10: chunks 4500–4999
Inserted batch 11: chunks 5000–5499
Inserted batch 12: chunks 5500–5999
Inserted batch 13: chunks 6000–6499
Inserted batch 14: chunks 6500–6701

Done. Total inserted: 6702


## Verify

Run a similarity search to confirm the store is working.

In [6]:
results = store.similarity_search(
    query="What are Apple's main revenue segments?",
    k=3,
    filter={"stock_symbol": "aapl"},
)

for i, doc in enumerate(results, 1):
    print(f"--- Result {i} ---")
    print(f"Section: {doc.metadata.get('section')} | doc_id: {doc.metadata.get('doc_id')}")
    print(doc.page_content[:300])
    print()

--- Result 1 ---
Section: Item 1. | doc_id: aapl-20250927
Cloud Services
The Companys cloud services store and keep customers content up-to-date and available across multiple Apple devices and Windows personal computers.
Digital Content
The Company operates various platforms, including the App Store, that allow customers to discover and download applicatio

--- Result 2 ---
Section: Item 1. | doc_id: aapl-20250927
Payment Services
The Company offers payment services, including Apple Card, a co-branded credit card, and Apple Pay, a cashless payment service.
Segments
The Company manages its business primarily on a geographic basis. The Companys reportable segments consist of the Americas, Europe, Greater China,

--- Result 3 ---
Section: Item 7. | doc_id: aapl-20250927
Greater China
Greater China net sales decreased during 2025 compared to 2024 primarily due to lower net sales of iPhone, partially offset by higher net sales of Mac.
Japan
Japan net sales increased during 2025 compared to 

## Filtered search example

Restrict to a specific ticker and section.

In [7]:
results = store.similarity_search(
    query="revenue and operating income",
    k=3,
    filter={"stock_symbol": "aapl", "section": "Item 7."},
)

for i, doc in enumerate(results, 1):
    print(f"--- Result {i} ---")
    print(f"Section: {doc.metadata.get('section')} | chunk: {doc.metadata.get('chunk')}")
    print(doc.page_content[:300])
    print()

--- Result 1 ---
Section: Item 7. | chunk: 12
Operating Expenses
Operating expenses for 2025, 2024 and 2023 were as follows (dollars in millions):
2025
Change
2024
Change
2023
Research and development
$
34,550
10
%
$
31,370
5
%
$
29,915
Percentage of total net sales
8
%
8
%
8
%
Selling, general and administrative
$
27,601
6
%
$
26,097
5
%
$
24,

--- Result 2 ---
Section: Item 7. | chunk: 7
Segment Operating Performance
The following table shows net sales by reportable segment for 2025, 2024 and 2023 (dollars in millions):
2025
Change
2024
Change
2023
Americas
$
178,353
7
%
$
167,045
3
%
$
162,560
Europe
111,032
10
%
101,328
7
%
94,294
Greater China
64,377
(4)
%
66,952
(8)
%
72,559
Jap

--- Result 3 ---
Section: Item 7. | chunk: 11
%
46.2
%
44.1
%
Products Gross Margin
Products gross margin increased during 2025 compared to 2024 primarily due to favorable costs and a different mix of products, partially offset by tariff costs.
Products gross margin percentage decreased during 2025 compa